# fastcolab

> Convenient access to the colab API

In [ ]:
#| hide
from fastcolab.core import *

Most developers know Google Colab as a hosted browser notebook for quick experiments. Underneath that browser UI, Google runs an on-demand CPU, GPU, and TPU compute platform with an official REST API at `colaboratory.googleapis.com`. You can provision machines, run Python scripts or multi-hour training jobs, transfer files, and tear runtimes down from code, without opening a browser or writing a notebook.

The cost is hard to beat. Google bundles 1,000 monthly Colab compute units into Google AI Ultra 5x ($99.99/month) and 2,000 into Ultra 20x ($199.99/month) alongside Gemini Ultra and cloud storage. Even at standalone Pay-As-You-Go prices ($0.10 per unit), Colab's accelerators cost a fraction of standard cloud GPU rates. A 96 GB RTX PRO 6000 runs around $0.67 to $0.89 an hour compared to $3.03 on Modal, and a single-chip Trillium TPU runs around $0.41 an hour compared to $2.70 on Google Cloud. If you already pay for a Google AI Ultra subscription, hundreds of GPU or TPU hours sit in your account every month.

`fastcolab` wraps Colab's REST API and runtime Jupyter proxy to make that compute practical from Python.

## Plans and compute units

All Colab plans and Google AI subscriptions credit the same account balance.

| Plan | Price | Monthly units | Included extras |
| --- | --- | --- | --- |
| Google AI Ultra 5x | $99.99/month | 1,000 | Gemini Ultra (5x), 20 TB storage, background execution |
| Google AI Ultra 20x | $199.99/month | 2,000 | Gemini Ultra (20x), 30 TB storage, background execution |
| Colab Pro+ | $49.99/month | 500 to 600 | Standalone subscription, background execution |
| Colab Pro / Pay-As-You-Go | $9.99 | 100 | Monthly subscription or one-time top-up |

Unused units roll over from month to month and expire 90 days after issue. Colab bills units for every minute a machine stays allocated, including idle periods. Delete each runtime as soon as work finishes.

## Choosing a runtime

Turn on High-RAM (`HIGHMEM`) on CPU and T4 runtimes. It quadruples RAM from roughly 13 GB to 51 GB and raises CPU cores from 2 to 8-12 for about 0.1 to 0.3 extra units per hour. Every higher accelerator provisions a High-RAM host automatically.

Two paid options are omitted because newer Colab hardware beats them on price and speed. A G4 (RTX PRO 6000 Blackwell) beats an A100 on every metric: 96 GB VRAM instead of 40 or 80 GB, more RAM, roughly 1.6x higher bfloat16 throughput, and lower hourly cost (~6.7-8.9 vs ~10-13+ units/hr). On paid credits, the Trillium TPU (`v6e-1`, ~4.08 units/hr) delivers roughly 4.5x the throughput of a `v5e-1` (~2.92 units/hr). Use `v5e-1` only on the free tier.

| Accelerator | VRAM | RAM | Relative speed | Units/hr | Hours per 1,000 units | Rate vs cloud on-demand |
| --- | --- | --- | --- | --- | --- | --- |
| CPU (High-RAM) | None | ~51 GB | - | ~0.4 | ~2,500 | ~$0.04/hr |
| T4 (High-RAM) | 16 GB | ~51 GB | ~0.5x L4 (FP16 only) | ~1.8-2.3 | ~450-550 | ~$0.18-$0.23/hr |
| L4 | 24 GB | ~51 GB | 1x (121 TFLOPS, 300 GB/s) | ~3.0 | ~330 | ~$0.30/hr |
| G4 (RTX PRO 6000) | 96 GB | ~177 GB | ~4x L4 (500 TFLOPS, 1.6 TB/s) | ~6.7-8.9 | ~110-150 | ~$0.67-$0.89/hr vs $3.03/hr (Modal) |
| H100 | 80 GB | ~167-200 GB | ~8x L4 (989 TFLOPS, 3.35 TB/s) | ~18.0 | ~55 | ~$1.80/hr vs $3.95/hr (Modal) |
| TPU v6e-1 (Trillium) | 32 GB | ~176+ GB | ~7.5x L4 (918 TFLOPS, 1.64 TB/s) | ~4.08 | ~245 | ~$0.41/hr vs $2.70/hr (GCP US) |

Pick the G4 as your default GPU. It has the largest VRAM on Colab, supports native FP8 and FP4, and costs roughly half as much per hour as an H100. Move up to the H100 when memory bandwidth or wall-clock speed dominates and the job fits in 80 GB. Pick the Trillium TPU (`v6e-1`) for the highest compute per credit when your code targets JAX, TensorFlow, or PyTorch XLA and fits in 32 GB.

Google does not publish fixed burn rates or guaranteed RAM allocations. Hourly rates vary slightly by region and load. Check live RAM inside any session with `free -h`.

Colab gives you single-accelerator virtual machines rather than elastic clusters. Each machine attaches one GPU or one TPU chip and caps out at 24 hours even with Ultra or Pro+ background execution. Google can also reclaim an instance when regional capacity tightens. Local disks (~70 to 100 GB) are wiped when a machine stops, so save checkpoints to Google Cloud Storage or Google Drive for long runs.

## Why the REST API is all you need

Google exposes two official interfaces under `colaboratory.googleapis.com`. The REST API handles `runtimes`, `runtimespecs`, and `getSubscription` in `v1beta`, plus long-running `operations` in `v1`. The hosted Model Context Protocol (MCP) server runs at `https://colaboratory.googleapis.com/mcp`.

`runtimes.create`, `runtimes.get`, and `runtimes.list` all return a `connectionInfo` object containing `url`, `token`, and `expireTime`. Sending `X-Colab-Runtime-Proxy-Token: {token}` to `connectionInfo.url` connects directly to the runtime's Jupyter server. Pairing REST with a Jupyter client such as `jupyasyncclient` gives you live streaming output, rich display objects such as plots, and plain HTTP file transfers.

The hosted MCP server adds four kernel tools (`create_session`, `list_sessions`, `delete_session`, `execute_code`) with stateless `executionId` reattachment across 600-second HTTP cutoffs. In exchange, MCP buffers output as text only and truncates large logs. It omits images and requires repeated 600-second requests for multi-hour runs. Driving REST plus the Jupyter proxy avoids those constraints.

Neither REST nor MCP exposes your live credit balance (`get_subscription` returns only the tier, such as `SUBSCRIPTION_TIER_PRO_PLUS`). To inspect live balance and burn rate, run `colab usage` from `google-colab-cli`. That command queries Colab's older `/tun/m/ccu-info` endpoint.

## Getting started

Give each developer their own External Google Cloud project with `gclientid`. Separate projects isolate API quotas per person and work with both personal `@gmail.com` Ultra accounts and Workspace accounts.

1. Create or update your External `gclientid` project (`gclientid` enables `colaboratory.googleapis.com` and declares the `colaboratory` scope automatically because `COLAB` is in `MAX`):

```sh
gclientid
```

2. Authorize the Google account that holds your Colab or Google AI Ultra credits:

```sh
gclientid-auth you@gmail.com --preset colab
```

`gclientid` stores tokens per email at `~/.config/gclientid/oauth-token-<email>.json`. Authorizing a personal `@gmail.com` account leaves existing Workspace tokens untouched.

Connect from Python with `fastgws` by loading the Colab discovery documents directly via `GWSApi.from_discovery_url`. Plain `GWSApi('colaboratory', version='v1beta', creds=creds)` returns an empty client with no operations because `colaboratory` is absent from Google's central discovery directory.

In [ ]:
#| eval: false
from fastgws.core import GWSApi, oauth_creds

creds = await oauth_creds(account='you@gmail.com', scopes=['https://www.googleapis.com/auth/colaboratory'])
colab = await GWSApi.from_discovery_url('https://colaboratory.googleapis.com/$discovery/rest?version=v1beta', creds=creds)
ops = await GWSApi.from_discovery_url('https://colaboratory.googleapis.com/$discovery/rest?version=v1', creds=creds)

Verify credentials and inspect eligible runtimes without allocating a machine:

In [ ]:
#| eval: false
await colab.v1beta.get_subscription(name='subscription')
await colab.runtimespecs.list()
await colab.runtimes.list()

Connect `jupyasyncclient` to `runtime.connectionInfo` with `X-Colab-Runtime-Proxy-Token`. Pass a `refresh_headers` callback to fetch a fresh token when reconnects outlive the 1-hour expiry:

In [ ]:
#| eval: false
from jupyasyncclient import JupyAsyncKernelClient

async def refresh():
    rt = await colab.runtimes.get(name=runtime.name)
    return {'X-Colab-Runtime-Proxy-Token': rt.connectionInfo.token}

hdrs = {'X-Colab-Runtime-Proxy-Token': runtime.connectionInfo.token}
kc = await JupyAsyncKernelClient.connect(runtime.connectionInfo.url, headers=hdrs, refresh_headers=refresh)

## Running long training jobs

Colab disconnects a runtime after roughly 90 minutes with an idle Jupyter kernel. If you spawn a detached `nohup` process and let the cell finish, the kernel sits idle and Colab can reap the machine even on Ultra or Pro+. Keep training inside an executing cell instead.

1. Create a runtime with `v1beta.runtimes.create` and wait on the returned operation via `v1.operations.wait` (or poll `v1.operations.get`).
2. Upload your training script via `JupyAsyncFilesClient` or `PUT {url}/api/contents/{path}`.
3. Open a kernel on `connectionInfo.url`, submit a cell that runs training with stdout and stderr flushed to `/content/train.log` and writes the exit code to `/content/train.exit` on completion, then close the local WebSocket with `aclose()` (`shutdown_kernel=False`). The remote kernel stays busy and holds the machine open for up to 24 hours with no client connected.
4. To check progress later, call `v1beta.runtimes.get` for a fresh proxy token, tail new log bytes via `GET {url}/files/content/train.log` with header `Range: bytes=<offset>-`, and check completion via `GET {url}/api/kernels/{kernel_id}` (`execution_state`) or `/content/train.exit`.
5. Sync checkpoints periodically to Google Cloud Storage or Google Drive from inside the training script, download remaining local files via `GET {url}/files/{path}`, and call `v1beta.runtimes.delete`.

## Developer reference

### REST endpoints and runtime specs

`v1beta` exposes `runtimes.create`, `runtimes.get`, `runtimes.list`, `runtimes.delete`, `runtimespecs.list`, and `v1beta.getSubscription`. `v1` exposes `operations.get`, `operations.list`, and `operations.wait`. Both versions require only the `https://www.googleapis.com/auth/colaboratory` scope (or `colaboratory.readonly`), without the `userinfo.email` and `cloud-platform` scopes required by `google-colab-cli`'s older session backend.

`runtimespecs.list` returns `(variant, accelerator, shape)` keys covering `DEFAULT`/`NONE` (`STANDARD`, `HIGHMEM`), `GPU` (`T4` `STANDARD` and `HIGHMEM`, `L4` `STANDARD`, `A100` `STANDARD` and `HIGHMEM`, `G4` `STANDARD`, `H100` `STANDARD`), and `TPU` (`V5E1` `STANDARD`, `V6E1` `STANDARD`).

### Proxy authentication, expiry, and Jupyter behavior

Passing `X-Colab-Runtime-Proxy-Token: {token}` alone authenticates both HTTP requests and verified `wss://` WebSocket connections to `connectionInfo.url`. You do not need the `colab-runtime-proxy-token` query parameter or the `X-Colab-Client-Agent` header. Proxy tokens expire after roughly one hour (`connectionInfo.expireTime`) and fail with HTTP 404 on both HTTP and WebSockets rather than 401. Call `v1beta.runtimes.get` to refresh the token before reconnecting.

Colab exposes kernelspecs `['julia', 'python3', 'ir']`. Passing `name="py"` to `POST /api/kernels` returns HTTP 500, while omitting `name` (or passing `"python3"`) selects Python 3. Colab speaks standard legacy JSON WebSocket frames (not `v1.kernel.websocket.jupyter.org`) and sends custom `colab_request` messages that clients can safely ignore. `JupyAsyncTerminalClient` is incompatible with Colab because it uses jupygate's `/api/terminals/{name}/channel` binary protocol instead of Jupyter's `/terminals/websocket/{name}`.

### File transfer protocol

Paths on `connectionInfo.url` omit the leading slash (`content/train.py` for `/content/train.py`). `JupyAsyncFilesClient` (`listing`, text and binary `write` and `read`, `mkdir`, `rename`, and `delete`) works unmodified against Colab's `/api/contents`.

For raw HTTP uploads, send `PUT {url}/api/contents/{path}` with `{"type": "file", "format": "base64", "content": "..."}` (or `"format": "text"`). Keep each request under 50 MB and send larger files as sequential pieces with `"chunk": 1`, `2`, and `"chunk": -1` on the final piece. Create missing parent directories one level per request with `{"type": "directory"}`. For raw downloads and incremental log tailing, `GET {url}/files/{path}` returns raw bytes and supports HTTP `Range` requests (HTTP 206).

### CLI fallback and live credit balance

For quick terminal checks or live compute unit balance without a Google Cloud project, install `google-colab-cli` (`uv tool install google-colab-cli`) and run `colab --auth=oauth2 whoami` and `colab --auth=oauth2 usage`. `colab usage` calls `colab_cli.client.Client.get_consumption_user_info()` against Colab's `/tun/m/ccu-info` tunnel endpoint. Its bundled OAuth client (`projects/764086051850`) cannot call `colaboratory.googleapis.com` directly without a separate quota project.